> ⚠️ Before running, set your `hub_model_id` and output paths for this experiment version.

> This template now targets `browndw/morphoseg-en` with v7 augmented-all-splits data (punctuation variants in train/validation/test) and full-run settings optimized for exact-match selection on validation. If your GPU is small, reduce batch size or gradient accumulation.

In [ ]:
!pip install -q evaluate

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

# MorphoSeg English ByT5 Training (Colab Template)

This notebook fine-tunes `google/byt5-small` on the `browndw/morphoseg-en` dataset.

In [ ]:
from huggingface_hub import notebook_login

In [ ]:
notebook_login()

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback

In [ ]:
import platform
import torch
import transformers
import datasets
import accelerate
import evaluate
import fsspec
import pyarrow
print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "accelerate": accelerate.__version__,
    "evaluate": evaluate.__version__,
    "fsspec": fsspec.__version__,
    "pyarrow": pyarrow.__version__,
})

In [ ]:
repo_id = "browndw/morphoseg-en"
model_name = "google/byt5-small"
raw_datasets = load_dataset(repo_id)
# Dataset has splits: train, validation, test
print(raw_datasets)

In [ ]:
def has_valid_segments(example):
    segments = example["segments"]
    return isinstance(segments, list) and len(segments) > 0 and all(str(seg).strip() for seg in segments)

raw_datasets = raw_datasets.filter(has_valid_segments)
raw_datasets

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
max_input_length = 40
max_target_length = 60

In [ ]:
def preprocess_function(examples):
    inputs = [str(word) for word in examples["word"]]
    targets = [" ".join(segments) for segments in examples["segments"]]
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding=False,
    )
    labels = tokenizer(
        targets,
        max_length=max_target_length,
        truncation=True,
        padding=False,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
tokenized_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
    load_from_cache_file=False,
)

# Run profile: set to "full" for final training, "smoke" for quick checks.
run_profile = "full"  # options: "full", "smoke"

if run_profile == "full":
    max_train_samples = 0
    max_eval_samples = 0
    max_test_samples = 0
else:
    max_train_samples = 40000
    max_eval_samples = 4000
    max_test_samples = 4000

if max_train_samples > 0:
    tokenized_datasets["train"] = tokenized_datasets["train"].select(
        range(min(max_train_samples, len(tokenized_datasets["train"])))
    )
if max_eval_samples > 0:
    tokenized_datasets["validation"] = tokenized_datasets["validation"].select(
        range(min(max_eval_samples, len(tokenized_datasets["validation"])))
    )
if max_test_samples > 0:
    tokenized_datasets["test"] = tokenized_datasets["test"].select(
        range(min(max_test_samples, len(tokenized_datasets["test"])))
    )

print({
    "run_profile": run_profile,
    "train_rows": len(tokenized_datasets["train"]),
    "validation_rows": len(tokenized_datasets["validation"]),
    "test_rows": len(tokenized_datasets["test"]),
})

tokenized_datasets

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    label_pad_token_id=-100,
    return_tensors="pt",
)

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
import numpy as np

def _normalize_token_sequence(seq):
    while isinstance(seq, (list, tuple)) and len(seq) > 0 and isinstance(seq[0], (list, tuple)):
        seq = seq[0]
    return list(seq) if isinstance(seq, (list, tuple)) else [int(seq)]

def _normalize_token_batch(batch):
    return [_normalize_token_sequence(seq) for seq in batch]

_eval_context = {
    "is_validation": False,
    "rows": None,
}

# Fast training-time metric: token-level exact match without decoding.
metric_mode = "token_exact_fast"
special_token_ids = set(int(x) for x in tokenizer.all_special_ids)

def _normalize_ids_for_match(seq, is_label=False):
    out = []
    for tok in seq:
        t = int(tok)
        if is_label and t == -100:
            continue
        if t in special_token_ids:
            continue
        out.append(t)
    return tuple(out)

raw_validation_labels = tokenized_datasets["validation"]["labels"]
normalized_validation_labels = _normalize_token_batch(raw_validation_labels)
cached_validation_label_ids = [
    _normalize_ids_for_match(seq, is_label=True)
    for seq in normalized_validation_labels
]
print({
    "metric_mode": metric_mode,
    "cached_validation_labels": len(cached_validation_label_ids),
})

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    if hasattr(preds, "ndim") and preds.ndim == 3:
        preds = np.argmax(preds, axis=-1)

    vocab_size = getattr(tokenizer, "vocab_size", None) or len(tokenizer)
    preds = np.clip(preds, 0, vocab_size - 1)

    pred_rows = _normalize_token_batch(preds.tolist())
    pred_norm = [_normalize_ids_for_match(seq, is_label=False) for seq in pred_rows]

    if _eval_context.get("is_validation", False):
        n = len(pred_norm)
        label_norm = cached_validation_label_ids[:n]
    else:
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        labels = np.clip(labels, 0, vocab_size - 1)
        label_rows = _normalize_token_batch(labels.tolist())
        label_norm = [_normalize_ids_for_match(seq, is_label=True) for seq in label_rows]

    exact_matches = [int(p == l) for p, l in zip(pred_norm, label_norm)]
    exact_match = float(np.mean(exact_matches)) if exact_matches else 0.0
    return {"exact_match": exact_match}

In [ ]:
# Runtime sanity check before long training.
import torch

require_cuda = True
cuda_ok = torch.cuda.is_available()
print({
    "torch_version": torch.__version__,
    "torch_cuda_version": torch.version.cuda,
    "cuda_available": cuda_ok,
    "cuda_device_count": torch.cuda.device_count(),
    "device_name": torch.cuda.get_device_name(0) if cuda_ok else None,
})

if require_cuda and not cuda_ok:
    raise RuntimeError(
        "CUDA is not available in this runtime. Switch Colab runtime to GPU and restart kernel."
    )

In [ ]:
import inspect
import math
import shutil
import torch
from pathlib import Path

output_dir = "/content/browndw-morphoseg-byt5-v7-fullrun"
hub_model_id = "browndw/morphoseg-en-byt5"  # optional, used only in final push cell

use_bf16 = False
if torch.cuda.is_available():
    major, _ = torch.cuda.get_device_capability(0)
    use_bf16 = major >= 8
use_fp16 = torch.cuda.is_available() and not use_bf16

args_kwargs = dict(
    output_dir=output_dir,
    learning_rate=3e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    num_train_epochs=24,
    predict_with_generate=True,
    generation_max_length=80,
    logging_steps=50,
    warmup_ratio=0.1,
    push_to_hub=False,
    seed=99,
    fp16=use_fp16,
    bf16=use_bf16,
    save_total_limit=1,
 )

sig_params = set(inspect.signature(Seq2SeqTrainingArguments.__init__).parameters)
if "evaluation_strategy" in sig_params:
    args_kwargs["evaluation_strategy"] = "epoch"
elif "eval_strategy" in sig_params:
    args_kwargs["eval_strategy"] = "epoch"
if "save_strategy" in sig_params:
    args_kwargs["save_strategy"] = "epoch"
if "logging_strategy" in sig_params:
    args_kwargs["logging_strategy"] = "steps"
if "report_to" in sig_params:
    args_kwargs["report_to"] = []
if "save_only_model" in sig_params:
    args_kwargs["save_only_model"] = True
if "save_safetensors" in sig_params:
    args_kwargs["save_safetensors"] = True

if "predict_with_generate" not in sig_params:
    args_kwargs.pop("predict_with_generate", None)
if "generation_max_length" not in sig_params:
    args_kwargs.pop("generation_max_length", None)
if "warmup_ratio" not in sig_params:
    args_kwargs.pop("warmup_ratio", None)
    args_kwargs["warmup_steps"] = 0
if "seed" not in sig_params:
    args_kwargs.pop("seed", None)
if "save_total_limit" not in sig_params:
    args_kwargs.pop("save_total_limit", None)
if "save_only_model" not in sig_params:
    args_kwargs.pop("save_only_model", None)
if "save_safetensors" not in sig_params:
    args_kwargs.pop("save_safetensors", None)
if "fp16" not in sig_params:
    args_kwargs.pop("fp16", None)
if "bf16" not in sig_params:
    args_kwargs.pop("bf16", None)

output_dir_path = Path(output_dir).expanduser().resolve()
output_dir_path.mkdir(parents=True, exist_ok=True)

total_b, used_b, free_b = shutil.disk_usage(output_dir_path)
free_gb = free_b / (1024 ** 3)
if free_gb < 8:
    raise RuntimeError(f"Low disk space for checkpoints: {free_gb:.2f} GB free")

train_rows = len(tokenized_datasets["train"])
validation_rows = len(tokenized_datasets["validation"])
test_rows = len(tokenized_datasets["test"])
if train_rows == 0:
    raise RuntimeError("Train split is empty after filtering/subsetting. Training cannot proceed.")
if validation_rows == 0:
    raise RuntimeError("Validation split is empty while evaluation_strategy is enabled.")

effective_batch = args_kwargs["per_device_train_batch_size"] * args_kwargs["gradient_accumulation_steps"]
steps_per_epoch = math.ceil(train_rows / effective_batch)
print({
    "output_dir": str(output_dir_path),
    "train_rows": train_rows,
    "validation_rows": validation_rows,
    "test_rows": test_rows,
    "effective_batch": effective_batch,
    "steps_per_epoch": steps_per_epoch,
    "metric_mode": metric_mode,
    "disk_free_gb": round(free_gb, 2),
})

training_args = Seq2SeqTrainingArguments(**args_kwargs)
training_args

In [ ]:
import inspect
from transformers import EarlyStoppingCallback

class FastMetricContextTrainer(Seq2SeqTrainer):
    def evaluate(self, eval_dataset=None, *args, **kwargs):
        global _eval_context
        ds = eval_dataset if eval_dataset is not None else getattr(self, "eval_dataset", None)
        rows = len(ds) if ds is not None else None

        # Use cached validation labels only for the trainer's validation eval path.
        is_validation_eval = eval_dataset is None or ds is tokenized_datasets["validation"]
        _eval_context = {"is_validation": bool(is_validation_eval), "rows": rows}

        out = super().evaluate(eval_dataset=eval_dataset, *args, **kwargs)
        _eval_context = {"is_validation": False, "rows": None}
        return out

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer_init_params = set(inspect.signature(Seq2SeqTrainer.__init__).parameters)
if "tokenizer" in trainer_init_params:
    trainer_kwargs["tokenizer"] = tokenizer
elif "processing_class" in trainer_init_params:
    trainer_kwargs["processing_class"] = tokenizer

trainer = FastMetricContextTrainer(**trainer_kwargs)
trainer.remove_callback(EarlyStoppingCallback)

print({
    "trainer_output_dir": trainer.args.output_dir,
    "train_rows": len(tokenized_datasets["train"]),
    "validation_rows": len(tokenized_datasets["validation"]),
    "predict_with_generate": trainer.args.predict_with_generate,
    "save_strategy": getattr(trainer.args, "save_strategy", None),
    "metric_mode": metric_mode,
})

In [ ]:
# Optional preflight eval before long training.
run_preflight_eval = False

if run_preflight_eval:
    import time
    preflight_n = min(64, len(tokenized_datasets["validation"]))
    preflight_ds = tokenized_datasets["validation"].select(range(preflight_n))
    t0_preflight = time.time()
    preflight_metrics = trainer.evaluate(eval_dataset=preflight_ds, metric_key_prefix="preflight")
    print({
        "preflight_eval_rows": preflight_n,
        "preflight_eval_seconds": round(time.time() - t0_preflight, 3),
        "preflight_exact_match": preflight_metrics.get("preflight_exact_match"),
    })
else:
    print("Preflight eval skipped.")

In [ ]:
import time
from pathlib import Path

t0_train = time.time()
train_result = trainer.train()
print({
    "train_runtime_seconds": round(time.time() - t0_train, 3),
    "global_step": int(getattr(trainer.state, "global_step", 0)),
    "epoch": float(getattr(trainer.state, "epoch", 0.0) or 0.0),
})
print(train_result)

final_dir = Path(training_args.output_dir) / "final_model"
final_dir.mkdir(parents=True, exist_ok=True)
t0 = time.time()
trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))
print({"final_save_dir": str(final_dir), "final_save_seconds": round(time.time() - t0, 3)})

In [ ]:
# Final test evaluation must use test labels (not cached validation labels).
_eval_context.update({"is_validation": False, "rows": len(tokenized_datasets["test"])})
metrics = trainer.evaluate(tokenized_datasets["test"], metric_key_prefix="test")
_eval_context.update({"is_validation": False, "rows": None})
metrics

In [ ]:
# Authenticate for Hub upload (interactive, no hard-coded token).
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Push only the finalized model folder so the Hub repo contains the intended final artifacts.
from pathlib import Path
from huggingface_hub import HfApi, upload_folder

final_dir = Path(training_args.output_dir) / "final_model"
if not final_dir.exists():
    raise RuntimeError(f"Final model directory not found: {final_dir}")

# Ensure a README/model card exists in the uploaded folder.
readme_path = final_dir / "README.md"
if not readme_path.exists():
    trainer.create_model_card(model_name=hub_model_id)
    root_readme = Path(training_args.output_dir) / "README.md"
    if root_readme.exists():
        readme_path.write_text(root_readme.read_text(encoding="utf-8"), encoding="utf-8")

api = HfApi()
api.create_repo(repo_id=hub_model_id, repo_type="model", exist_ok=True)

commit_info = upload_folder(
    repo_id=hub_model_id,
    repo_type="model",
    folder_path=str(final_dir),
    commit_message="Upload final model artifacts",
)
print(commit_info)